In [11]:
import os
import glob
import datetime

import numpy as np
import pandas as pd

import jax
import numpyro

import hssm
import arviz as az
from scipy.stats import gaussian_kde

import matplotlib.pyplot as plt
import seaborn as sns

import sqlite3

In [12]:
db_path = "hssm_fits.sqlite"
with sqlite3.connect(db_path) as conn:
    # Get all tables that exist
    tables = pd.read_sql(
        "SELECT name FROM sqlite_master WHERE type='table';",
        conn
    )
    print("Tables in database:")
    print(tables)

Tables in database:
         name
0  ddm_mod_th
1   ddm_mod_v
2    ddm_pure


In [13]:
with sqlite3.connect(db_path) as conn:
    df_ddm_mod_th = pd.read_sql("SELECT * FROM ddm_mod_th;", conn)
    th_df = pd.DataFrame(df_ddm_mod_th)

    df_ddm_mod_v = pd.read_sql("SELECT * FROM ddm_mod_v;", conn)
    v_df = pd.DataFrame(df_ddm_mod_v)

    df_ddm_pure = pd.read_sql("SELECT * FROM ddm_pure;", conn)
    pure_df = pd.DataFrame(df_ddm_pure)

In [14]:
pure_df[pure_df.participant_id == 708]

,param,mean,sd,hdi_3%,hdi_97%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat,participant_id
432,v,0.595,0.178,0.283,0.923,0.009,0.006,430.0,402.0,1.0,708
433,t,0.614,0.023,0.571,0.658,0.001,0.001,361.0,469.0,1.0,708
434,a,0.872,0.047,0.783,0.956,0.002,0.002,407.0,487.0,1.0,708
435,z,0.494,0.043,0.415,0.570,0.002,0.002,393.0,470.0,1.0,708


In [15]:
v_df[v_df['participant_id'] == 708]

,param,mean,sd,hdi_3%,hdi_97%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat,participant_id,timestamp
0,v_Intercept,0.478,0.148,0.194,0.746,0.006,0.004,672.0,625.0,1.00,708,2026-01-30T17:44:28.423188
1,v_X,0.084,0.124,-0.172,0.284,0.005,0.004,655.0,596.0,1.00,708,2026-01-30T17:44:28.423188
2,z,0.508,0.038,0.437,0.579,0.002,0.001,545.0,512.0,1.01,708,2026-01-30T17:44:28.423188
3,a,0.866,0.047,0.790,0.960,0.002,0.001,551.0,579.0,1.00,708,2026-01-30T17:44:28.423188
4,t,0.619,0.022,0.577,0.656,0.001,0.001,421.0,408.0,1.00,708,2026-01-30T17:44:28.423188


In [16]:
th_df[th_df['participant_id']==708].head(10)

,param,mean,sd,hdi_3%,hdi_97%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat,participant_id,timestamp
0,v_Intercept,0.535,0.139,0.291,0.820,0.006,0.004,517.0,598.0,1.00,708,2026-01-26T23:32:48.627410
1,a_Intercept,0.845,0.059,0.737,0.952,0.002,0.002,596.0,690.0,1.00,708,2026-01-26T23:32:48.627410
2,a_X,0.076,0.074,-0.071,0.210,0.003,0.002,705.0,586.0,1.01,708,2026-01-26T23:32:48.627410
3,t,0.615,0.025,0.567,0.659,0.001,0.001,424.0,386.0,1.00,708,2026-01-26T23:32:48.627410
4,z,0.508,0.040,0.436,0.583,0.002,0.001,589.0,636.0,1.00,708,2026-01-26T23:32:48.627410
5,a[0],0.845,0.059,0.737,0.952,0.002,0.002,596.0,690.0,1.00,708,2026-01-26T23:32:48.627410
6,a[1],0.845,0.059,0.737,0.952,0.002,0.002,596.0,690.0,1.00,708,2026-01-26T23:32:48.627410
7,a[2],0.845,0.059,0.737,0.952,0.002,0.002,596.0,690.0,1.00,708,2026-01-26T23:32:48.627410
8,a[3],0.845,0.059,0.737,0.952,0.002,0.002,596.0,690.0,1.00,708,2026-01-26T23:32:48.627410
9,a[4],0.845,0.059,0.737,0.952,0.002,0.002,596.0,690.0,1.00,708,2026-01-26T23:32:48.627410


In [17]:
len(pure_df.participant_id.unique()) == len(th_df.participant_id.unique()) == len(v_df.participant_id.unique())

True

# models setup

In [18]:
from hssm_utils import get_fitted_participants, as_trialwise
from hssm_models import simulate_participant_ddm

Setting PyTensor floatX type to float32.
Setting "jax_enable_x64" to False. If this is not intended, please set `jax` to False.


# simulate all participants

In [19]:
pure_simulations = []
th_simulations = []
v_simulations = []

for pid in pure_df.participant_id.unique():

    rng = np.random.default_rng(seed=pid)
    bonuses = rng.binomial(1, 0.5, 300)

    pure_sim = simulate_participant_ddm(pid, pure_df, model='pure', X=bonuses)
    th_sim = simulate_participant_ddm(pid, th_df, model='th', X=bonuses)
    v_sim = simulate_participant_ddm(pid, v_df, model='v', X=bonuses)
    
    pure_simulations.append(pure_sim)
    th_simulations.append(th_sim)
    v_simulations.append(v_sim)


pure_simulations_df = pd.concat(pure_simulations, ignore_index=True)
th_simulations_df   = pd.concat(th_simulations,   ignore_index=True)
v_simulations_df    = pd.concat(v_simulations,    ignore_index=True)

In [20]:
pure_simulations_df

,rt,response,participant_id,X
0,1.650991,-1.0,935,0
1,1.333995,1.0,935,0
2,1.418993,-1.0,935,0
3,2.357024,-1.0,935,0
4,2.042010,1.0,935,0
...,...,...,...,...
102895,1.529994,1.0,1111,0
102896,1.053996,1.0,1111,1
102897,4.456893,-1.0,1111,0
102898,1.178994,1.0,1111,1


In [28]:
import sqlite3

# connect/create database
conn = sqlite3.connect("hssm_sims.sqlite")

# write each model into its own table
pure_simulations_df.to_sql("pure_ddm", conn, if_exists="replace", index=False)
v_simulations_df.to_sql("v_ddm", conn, if_exists="replace", index=False)
th_simulations_df.to_sql("th_ddm", conn, if_exists="replace", index=False)

# close connection
conn.close()